# 04 - Patch tiling and windowed stitching

This notebook confirms the tiling and stitching machinery: how `Patcher.build` lays an overlapping patch grid over a region, the shape of the blending windows produced by `CubeStitcher.make_patch_window`, and that adding overlapping patches and calling `finalize_cube` recovers a known field. A constant field must come back flat, and a smooth ramp must be reproduced under overlap-add weighting.

Modules exercised: `pipelines.dataset_pipeline.spatial.Patcher` / `GridInfo`, `pipelines.inference_pipeline.predictor.CubeStitcher`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

import matplotlib.pyplot as plt
import numpy             as np
import torch

NP_SEED    = 0
TORCH_SEED = 0
np.random.seed(NP_SEED)
torch.manual_seed(TORCH_SEED)
torch.use_deterministic_algorithms(False)

plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (6.0, 4.0)
plt.rcParams["axes.grid"]   = True
plt.rcParams["grid.alpha"]  = 0.3

print("Repository root :", REPO_ROOT)
print("Torch device    : cpu (CPU only, no checkpoints required)")


## Build a patch grid over a small region

We tile a `20 x 20` region with `8 x 8` patches at stride `4`, giving a 50 percent overlap. The grid reports its patch count, padding, and padded size.

In [ ]:
from pipelines.dataset_pipeline.spatial import Patcher

spatial_size = (20, 20)
patch_size   = (8, 8)
stride       = 4

patcher = Patcher.build(spatial_size=spatial_size, patch_size=patch_size, stride=stride)
grid    = patcher.grid

print('patches     :', grid.number_of_patches, f'({grid.n_v} x {grid.n_h})')
print('patch size  :', grid.patch_size, 'stride', grid.stride)
print('padding     : top/bot', (grid.pad_top, grid.pad_bot), 'left/right', (grid.pad_left, grid.pad_right))
print('spatial/pad :', grid.spatial_size, '->', grid.padded_size)


## Blending window shapes

`make_patch_window` builds the per-patch weighting used in overlap-add. We compare the three kinds: `uniform` (flat), `hann` (smooth taper to the edges), and `triangular`. The Hann and triangular windows are floored at `1e-3` to keep the overlap-add denominator strictly positive.

In [ ]:
from pipelines.inference_pipeline.predictor import CubeStitcher

kinds   = ['uniform', 'hann', 'triangular']
windows = {k: CubeStitcher.make_patch_window(patch_size, kind=k) for k in kinds}

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4))
for ax, k in zip(axes, kinds):
    im = ax.imshow(windows[k], cmap='magma', vmin=0, vmax=1)
    ax.set_title(f'{k}\nmin={windows[k].min():.3g}')
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle('CubeStitcher.make_patch_window kinds')
fig.tight_layout()
plt.show()


## Constant field round-trips exactly

We feed every patch a constant value of `1.0`. After overlap-add normalisation by the accumulated weights, `finalize_cube` must return a field that is `1.0` everywhere, regardless of window kind, because the normalisation divides out the weighting.

In [ ]:
ph, pw = grid.patch_size

def stitch_constant(kind, value=1.0):
    st = CubeStitcher(grid=grid, n_channels=1, window_kind=kind, dtype='float32')
    for idx in range(grid.number_of_patches):
        st.add_patch(idx, np.full((1, ph, pw), value, dtype=np.float32))
    return st.finalize_cube()[0]

for kind in kinds:
    cube = stitch_constant(kind)
    print(f'{kind:11s}: min={cube.min():.6f} max={cube.max():.6f} shape={cube.shape}')


## Smooth ramp is reproduced under overlap-add

A harder test: we build a global horizontal ramp, extract the correct sub-block for every patch position, stitch it back with the Hann window, and compare to the original ramp. The overlap-add of a smooth field should match closely.

In [ ]:
H, W   = grid.spatial_size
ramp   = np.tile(np.linspace(0.0, 1.0, W, dtype=np.float32), (H, 1))

st = CubeStitcher(grid=grid, n_channels=1, window_kind='hann', dtype='float32')
for idx in range(grid.number_of_patches):
    iv, ih = divmod(idx, grid.n_h)
    v0 = iv * grid.stride - grid.pad_top
    h0 = ih * grid.stride - grid.pad_left
    patch = np.zeros((1, ph, pw), dtype=np.float32)
    for r in range(ph):
        for c in range(pw):
            gv, gh = v0 + r, h0 + c
            gv = min(max(gv, 0), H - 1)
            gh = min(max(gh, 0), W - 1)
            patch[0, r, c] = ramp[gv, gh]
    st.add_patch(idx, patch)

recovered = st.finalize_cube()[0]
err = np.abs(recovered - ramp)
print('max abs error :', float(err.max()))
print('mean abs error:', float(err.mean()))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.4))
for ax, (data, title) in zip(axes, [(ramp, 'original ramp'),
                                     (recovered, 'stitched (hann)'),
                                     (err, '|stitched - original|')]):
    cmap = 'magma' if title.startswith('|') else 'viridis'
    im = ax.imshow(data, cmap=cmap)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()


## Expected visual outcome

The window panels should show a flat uniform window, a centrally peaked Hann window, and a pyramidal triangular window, all with a small positive floor. The constant field must return `1.0` everywhere for all kinds. The ramp must be recovered with a near-zero error map, demonstrating that overlap-add stitching is weight-normalised and unbiased on smooth fields.